In [1]:
from collections import defaultdict, Counter

import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from pymutspec.annotation import CodonAnnotation
from scipy.stats import shapiro, pearsonr, spearmanr
from Bio import SeqIO
from pymutspec.draw import plot_mutspec12

from aminoacidshift import (
    amino_acid_codes, get_equilibrium_probabilities, 
    prepare_aa_subst, prepare_exp_aa_subst, plot_aa_eq_freqs,
    calc_metrics, plot_obs_vs_exp,
    get_equilibrium_freqs, get_equilibrium_freqs_rnd
)

coda = CodonAnnotation(2)


## scatterplot

In [2]:
def read_aa_counts_from_gb(path='../../data/reference/NC_012920.1.gb'):
    refseq = SeqIO.parse(path, 'genbank')
    refseq = list(refseq)[0]

    aa_freqs = dict()
    for fea in refseq.features:
        if fea.type == 'CDS':
            # print(fea.qualifiers['gene'])
            # print(fea.qualifiers['translation'])
            # print('---')
            aa_freqs[fea.qualifiers['gene'][0]] = dict(Counter(fea.qualifiers['translation'][0]))
    df = pd.DataFrame(aa_freqs).T.rename(columns=amino_acid_codes)
    return df

def plot_obs_vs_exp_custom(
        aa_subst, save_path=None, ax=None, show=True, 
        text='', text_x=-2.2, text_y=-4., metrics=None,
    ):
    aa_subst = aa_subst.copy()
    aa_subst['nobs_freqs_log'] = np.log10(aa_subst['nobs_freqs'])
    aa_subst['nexp_freqs_log'] = np.log10(aa_subst['nexp_freqs'])

    if ax is None:
        plt.figure(figsize=(5, 5))
        ax = plt.gca()

    ax = sns.scatterplot(aa_subst, ax=ax,
                x='nobs_freqs_log', y='nexp_freqs_log', 
                color='blue', alpha=0.7)

    # ticks = np.log10(np.array([1e-4, 1e-3, 1e-2, 1e-1]))
    ticks = np.linspace(-4, -1, 4)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    formatter = lambda x, pos: '10$^{' + str(int(x)) + '}$'
    ax.get_xaxis().set_major_formatter(formatter)
    ax.get_yaxis().set_major_formatter(formatter)
    # ax.get_xaxis().set_minor_locator(LogLocator(-10, 'auto'))

    r2 = 0 if metrics is None else metrics['r2']
    spearman = spearmanr(aa_subst['nobs_freqs'], aa_subst['nexp_freqs']).correlation

    ax.text(text_x, text_y, 
            f"{text}\n$R^2$={r2:.2f}\nspearman r={spearman:.2f}", 
            fontsize=12)
    ax.plot([-4, -1], [-4, -1], color='black', alpha=0.5, linestyle='--', label='y=x')
    ax.set_xlabel('Observed AA substitutions frequencies', fontsize=13)
    ax.set_ylabel('Expected AA substitutions frequencies', fontsize=13)
    ax.legend(loc='upper left', fontsize=12)
    if save_path:
        plt.savefig(save_path)
    if show:
        plt.show()


In [3]:
spectra = pd.read_csv('../../data/external/human_megatree_genes_spectra.csv')
spectra['MutRate'] = (spectra.ObsNum / spectra.ObsNum.sum()) / (spectra.ExpNum / spectra.ExpNum.sum())
spectrum_mean_ex_nd6 = spectra.query('Gene != "ND6"').groupby('Mut').MutRate.mean().rename('rate').reset_index()

# read amino acid freqs from protein files
aa_freqs = read_aa_counts_from_gb()
aa_freqs_ex_nd6 = (aa_freqs.sum() - aa_freqs.loc['ND6'].fillna(0)).to_dict()

obs = pd.read_csv('../../data/external/raw_human_megatree.tsv', sep='\t')
obs_ex_nd6 = obs[(obs.TypeRef == 'CDS') & (obs.Label == 0) & (obs.StrandRef == 1)]\
    .rename(columns={'Aa1': 'aa1', 'Aa2': 'aa2', 'ProbaFull': 'count'})


In [4]:
plot_mutspec12(spectra.query('Gene == "ND6"'), title='ND6', ylabel='Mutation rate', )

/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/pymutspec/draw/spectra.py:294: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Axes: title={'center': 'ND6'}, ylabel='Mutation rate'>

In [5]:
plot_mutspec12(spectra.query('Gene == "CYTB"'), title='CYTB', ylabel='Mutation rate', )

/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/pymutspec/draw/spectra.py:294: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Axes: title={'center': 'CYTB'}, ylabel='Mutation rate'>

In [6]:
exp_aa_subst, _ = prepare_exp_aa_subst(spectrum_mean_ex_nd6, 'rate', 2)
aa_subst = prepare_aa_subst(obs_ex_nd6, exp_aa_subst, aa_freqs_ex_nd6)
cur_metrics = calc_metrics(aa_subst)

plot_obs_vs_exp_custom(aa_subst, show=False, metrics=cur_metrics, text_x=-2.1, text_y=-4.)

plt.tight_layout()
plt.show()



/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


/tmp/ipykernel_85953/1903634063.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## fit quality

In [7]:
spectra_human_megatree = pd.read_csv(
    '../../data/external/human_megatree_genes_spectra.csv', index_col=0)
spectra_human_megatree.head()


,Mut,ObsNum,ExpNum,MutSpec,MutSpec_q05,MutSpec_median,MutSpec_q95
Gene,,,,,,,
ATP6,A>C,4.82915,49.0,0.001958,0.000683,0.001606,0.003860
ATP6,A>G,493.37400,79.0,0.124053,0.115703,0.123299,0.133501
ATP6,A>T,14.66790,49.0,0.005946,0.003896,0.005667,0.008291
ATP6,C>A,25.38110,49.0,0.010289,0.007022,0.010073,0.013041
ATP6,C>G,6.78884,49.0,0.002752,0.000820,0.002442,0.004783


In [8]:
genes_order = ['ND1', 'ND2', 'COX1', 'COX2', 'ATP8', 'ATP6', 'COX3',
       'ND3', 'ND4L', 'ND4', 'ND5', 'CYTB']

# sns.relplot(spectra_human_megatree.loc[genes_order].reset_index(), 
#             hue='Mut', y='MutSpec',
#             x='Gene', palette='Set2',
#             height=5, aspect=2,
#             kind='line', hue_order=['A>G', 'C>T', 'G>A', 'T>C'],
#             )

In [9]:
# obs_raw = pd.read_csv('../../data/external/raw_human_megatree.tsv', sep='\t')
obs_raw = pd.read_csv('../../data/external/raw_human_megatree.tsv', sep='\t')
obs = obs_raw[(obs_raw.TypeRef == 'CDS') & (obs_raw.Label == 0)]\
    .rename(columns={'Aa1': 'aa1', 'Aa2': 'aa2', 'ProbaFull': 'count'})
obs.groupby('GeneRef')['count'].sum()#.reset_index().sort_values('ProbaFull', ascending=False)


GeneRef
ATP6    3007.788022
ATP8     467.372545
COX1    1688.076914
COX2    1067.319400
COX3    1339.209707
CYTB    3108.850859
ND1     1993.161602
ND2     1525.177955
ND3      558.374301
ND4     1477.111861
ND4L     192.257186
ND5     3380.719033
ND6     1074.930743
Name: count, dtype: float64

In [10]:
obs_raw.groupby('Label').ProbaFull.sum()

Label
-4       75.309187
-3       77.834825
-2       85.335789
-1       31.600353
 0    74130.064443
 1    17753.729651
 2    19535.898506
Name: ProbaFull, dtype: float64

## Check gradient

In [11]:
sites = obs_raw.SiteRef.drop_duplicates().sort_values()
plt.hist(sites, bins=500);

In [12]:
ref_annot = pd.read_csv('../../data/external/hum_ref_annot.csv')
ref_annot


,Pos,Nuc,Context,Strand,Type,GeneName,PosInGene,PosInCodon,Codon,TBSS
0,1,G,NaN,-1,D-loop,NaN,-1,-1,NaN,NaN
1,2,A,GAT,-1,D-loop,NaN,-1,-1,NaN,NaN
2,3,T,ATC,-1,D-loop,NaN,-1,-1,NaN,NaN
3,4,C,TCA,-1,D-loop,NaN,-1,-1,NaN,NaN
4,5,A,CAC,-1,D-loop,NaN,-1,-1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
16564,16565,C,ACG,-1,D-loop,NaN,-1,-1,NaN,NaN
16565,16566,G,CGA,-1,D-loop,NaN,-1,-1,NaN,NaN
16566,16567,A,GAT,-1,D-loop,NaN,-1,-1,NaN,NaN
16567,16568,T,ATG,-1,D-loop,NaN,-1,-1,NaN,NaN


In [13]:
_revcomper = {'C>T':'G>A', 'G>A':'C>T', 'A>G':'T>C', 
              'T>C':'A>G', 'A>T':'T>A', 'T>A':'A>T', 
              'C>A':'G>T', 'G>T':'C>A', 'A>C':'T>G', 
              'T>G':'A>C', 'C>G':'G>C', 'G>C':'C>G'}

In [14]:
obs_syn = obs_raw.query('Label > 0 & GeneRef != "ND6"').copy()
obs_syn['SiteRef'] = obs_syn['SiteRef'].astype(int)

window = 200
step = window

# sliding window on genome positions
i = 0
data = []
for site in range(int(obs_raw.SiteRef.min()), int(obs_raw.SiteRef.max())-window+1, step):
    start = site
    end = site + window

    cur_obs = obs_syn.query('SiteRef >= @start and SiteRef < @end')
    nsyn = cur_obs.ProbaFull.sum()
    nsites = cur_obs.SiteRef.nunique()
    if nsyn < 25:
        continue

    gene = cur_obs.groupby('GeneRef').ProbaFull.sum().sort_values().tail(1).index.item()

    window_annot = ref_annot.query('@start <= Pos < @end and Type == "CDS"')
    first_pic = window_annot.iloc[0]['PosInCodon']
    last_pic = window_annot.iloc[-1]['PosInCodon']
    if first_pic == 2:
        window_annot = window_annot.iloc[2:]
    elif first_pic == 3:
        window_annot = window_annot.iloc[1:]

    if last_pic == 1:
        window_annot = window_annot.iloc[:-1]
    elif last_pic == 2:
        window_annot = window_annot.iloc[:-2]

    # drop incomplete stop-codons
    window_annot = window_annot[window_annot.Codon.str.len() == 3]

    exp_syn12dct = dict(coda.collect_exp_mut_freqs(window_annot.Nuc.values)[0]['syn'])

    cur_ms12 = pd.DataFrame([exp_syn12dct], index=['ExpNum']).T.reset_index(names=['Mut'])
    cur_ms12['ObsNum'] = cur_ms12.Mut.map(cur_obs.groupby('sbs12').ProbaFull.sum().rename('ObsNum')).fillna(0)
    cur_ms12['rate'] = (cur_ms12.ObsNum / cur_ms12.ObsNum.sum()) / (cur_ms12.ExpNum / cur_ms12.ExpNum.sum())
    # cur_ms12['rate'] /= cur_ms12['rate'].sum()
    cur_ms12 = cur_ms12.assign(start=start, end=end, nsyn=nsyn, gene=gene, nsites=nsites, window_id=i+1)
    data.append(cur_ms12)

    i += 1
    # if i > 20:
    #     break

df_grad = pd.concat(data, ignore_index=True) #.set_index(['start', 'end', 'nsyn', 'gene'])
# df_grad['rate_sum'] = df_grad[_revcomper.keys()].sum(1)
# for sbs in _revcomper.keys():
#     df_grad[sbs] /= df_grad['rate_sum']
df_grad.columns.name = None
df_grad.head()

,Mut,ExpNum,ObsNum,rate,start,end,nsyn,gene,nsites,window_id
0,A>G,12,51.420051,1.301445,3203,3403,227.182366,ND1,35,1
1,C>T,15,56.259984,1.139155,3203,3403,227.182366,ND1,35,1
2,C>G,8,0.000000,0.000000,3203,3403,227.182366,ND1,35,1
3,C>A,8,0.985479,0.037414,3203,3403,227.182366,ND1,35,1
4,G>A,3,23.433272,2.372390,3203,3403,227.182366,ND1,35,1


In [15]:
df_grad.query('window_id == 21')

,Mut,ExpNum,ObsNum,rate,start,end,nsyn,gene,nsites,window_id
233,C>T,6,21.083763,0.772519,7403,7603,141.009803,COX1,16,21
234,C>G,2,0.000000,0.000000,7403,7603,141.009803,COX1,16,21
235,C>A,2,1.941130,0.213372,7403,7603,141.009803,COX1,16,21
236,A>G,8,104.395575,2.868828,7403,7603,141.009803,COX1,16,21
237,A>C,4,0.000000,0.000000,7403,7603,141.009803,COX1,16,21
238,A>T,4,0.000000,0.000000,7403,7603,141.009803,COX1,16,21
239,T>C,2,3.978768,0.437352,7403,7603,141.009803,COX1,16,21
240,T>A,1,0.000000,0.000000,7403,7603,141.009803,COX1,16,21
241,T>G,1,0.000000,0.000000,7403,7603,141.009803,COX1,16,21
242,G>A,1,9.610568,2.112815,7403,7603,141.009803,COX1,16,21


In [16]:
df_grad.query('Mut == "T>C" & rate < 1.5')

,Mut,ExpNum,ObsNum,rate,start,end,nsyn,gene,nsites,window_id
239,T>C,2,3.978768,0.437352,7403,7603,141.009803,COX1,16,21


In [17]:
df_grad.nsites.hist(bins=25)

<Axes: xlabel='Observed AA substitutions frequencies', ylabel='Expected AA substitutions frequencies'>

In [18]:
fig, axs = plt.subplots(2, 1, figsize=(8, 10))

sns.lineplot(
    df_grad.query('nsites > 30'), 
    x='start', y='rate', hue='Mut', 
    hue_order=['C>T', 'G>A'],
    ax=axs[0]
)
_  = df_grad.query('nsites > 30 & Mut == "G>A" & end > 5904')
sns.regplot(_, x='start', y='rate', color='k', scatter=False, ax=axs[0])
print(shapiro(_.start), shapiro(_.rate))
print(pearsonr(_.start, _.rate))

_ = df_grad.query('nsites > 30 & Mut == "G>A" & end < 5904')
sns.regplot(_, x='start', y='rate', color='k', scatter=False, line_kws={'linestyle':'--'}, ax=axs[0])
print(shapiro(_.start), shapiro(_.rate))
print(pearsonr(_.start, _.rate))
print()

axs[0].set_ylabel('Частота мутаций', fontsize=12)
axs[0].set_xlabel('Позиция генома', fontsize=12)
axs[0].legend(loc='upper center', fontsize=12)


sns.lineplot(
    df_grad.query('nsites > 30'), 
    x='start', y='rate', hue='Mut', 
    hue_order=['T>C', 'A>G'],
    ax=axs[1]
)

_  = df_grad.query('nsites > 30 & Mut == "T>C" & end > 5904')
sns.regplot(_, x='start', y='rate', color='k', scatter=False, ax=axs[1])
print(shapiro(_.start), shapiro(_.rate))
print(pearsonr(_.start, _.rate))

_ = df_grad.query('nsites > 30 & Mut == "T>C" & end < 5904')
sns.regplot(_, x='start', y='rate', color='k', scatter=False, line_kws={'linestyle':'--'}, ax=axs[1])
print(shapiro(_.start), shapiro(_.rate))
print(pearsonr(_.start, _.rate))

axs[1].set_ylabel('Частота мутаций', fontsize=12)
axs[1].set_xlabel('Позиция генома', fontsize=12)
axs[1].legend(loc='upper center', fontsize=12)
plt.savefig('human_germline_mutation_rate_sliding_window.pdf')
plt.show()

ShapiroResult(statistic=np.float64(0.9632533838676961), pvalue=np.float64(0.17203518522419964)) ShapiroResult(statistic=np.float64(0.9714238359775622), pvalue=np.float64(0.33947088916171614))
PearsonRResult(statistic=np.float64(0.42659782301283977), pvalue=np.float64(0.00388111430161985))
ShapiroResult(statistic=np.float64(0.9511814744258105), pvalue=np.float64(0.6590831358918918)) ShapiroResult(statistic=np.float64(0.9415694445956814), pvalue=np.float64(0.5391213950227317))
PearsonRResult(statistic=np.float64(0.1383394988420186), pvalue=np.float64(0.6850054563008904))

ShapiroResult(statistic=np.float64(0.9611970892090832), pvalue=np.float64(0.12761869422036615)) ShapiroResult(statistic=np.float64(0.9679010357243094), pvalue=np.float64(0.23110483535541032))
PearsonRResult(statistic=np.float64(0.4129261093625814), pvalue=np.float64(0.004344377176872552))


ShapiroResult(statistic=np.float64(0.9511814744258105), pvalue=np.float64(0.6590831358918918)) ShapiroResult(statistic=np.float64(0.930910538725998), pvalue=np.float64(0.42013806661117015))
PearsonRResult(statistic=np.float64(-0.4441974023978272), pvalue=np.float64(0.17108549710822332))


/tmp/ipykernel_85953/3223200041.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
ref_annot.query('Type == "CDS"').groupby('GeneName').Pos.agg(['min', 'max']).sort_values('min')

,min,max
GeneName,,
ND1,3307,4262
ND2,4470,5511
COX1,5904,7445
COX2,7586,8269
ATP8,8366,8526
ATP6,8527,9206
COX3,9207,9990
ND3,10059,10404
ND4L,10470,10759


## Explore gene fitting

In [20]:
from Bio import SeqIO

refseq = SeqIO.parse('../../data/reference/NC_012920.1.gb', 'genbank')
refseq = list(refseq)[0]


In [21]:
from collections import Counter

gene_lens = dict()
aa_freqs = dict()
for fea in refseq.features:
    if fea.type == 'CDS':
        # print(fea.qualifiers['gene'])
        # print(fea.qualifiers['translation'])
        # print('---')
        aa_freqs[fea.qualifiers['gene'][0]] = dict(Counter(fea.qualifiers['translation'][0]))
        gene_lens[fea.qualifiers['gene'][0]] = len(fea.qualifiers['translation'][0])

In [22]:
pd.DataFrame(aa_freqs).T.fillna(0).astype(int).rename(columns=amino_acid_codes)

,Met,Pro,Ala,Asn,Leu,Ile,Val,Phe,Thr,Glu,Arg,Lys,Gly,Tyr,Gln,Asp,Ser,Trp,His,Cys
ND1,16,22,27,13,63,23,10,16,35,11,7,7,12,14,6,4,21,9,2,0
ND2,25,23,20,20,64,31,8,15,43,6,4,12,13,10,10,0,28,11,4,0
COX1,32,29,40,17,62,38,36,41,34,10,8,10,46,22,6,15,32,16,18,1
COX2,10,15,14,7,33,22,13,10,21,11,6,4,11,9,7,11,10,4,6,3
ATP8,6,11,0,5,10,3,1,1,8,1,0,7,0,2,3,0,4,3,2,1
ATP6,12,14,19,11,44,29,8,9,26,3,4,6,8,3,7,1,13,3,6,0
COX3,11,13,15,6,34,14,13,23,24,7,5,3,19,11,9,3,21,12,17,1
ND3,8,8,8,4,28,9,3,8,7,5,1,3,3,3,3,3,6,4,0,1
ND4L,10,2,9,6,23,7,6,3,5,2,1,0,4,4,1,1,8,0,3,3
ND4,27,23,26,23,96,39,13,20,48,9,9,11,17,13,10,3,43,13,13,3


In [23]:
metrics = pd.read_csv('./megatree_fit_metrics.csv', index_col=0)
metrics['gene_len'] = metrics.index.map(gene_lens)
metrics['mut_count_rel'] = metrics['mut_count'] / metrics['gene_len']
metrics[['r2', 'pearson_corr_clr', 'mut_count', 'mut_type_count', 'gene_len', 'mut_count_rel']]\
    .sort_values('r2', ascending=False)

,r2,pearson_corr_clr,mut_count,mut_type_count,gene_len,mut_count_rel
gene,,,,,,
ATP6,0.527012,0.739845,3007.790,91,226,13.308805
CYTB,0.460422,0.680245,3108.850,97,380,8.181184
COX1,0.376108,0.617446,1688.080,98,513,3.290604
COX2,0.371417,0.609182,1067.320,67,227,4.701850
ND2,0.356389,0.595902,1515.380,78,347,4.367089
ND6,0.317045,0.573904,1074.930,70,174,6.177759
COX3,0.287122,0.539322,1339.210,83,261,5.131073
ATP8,0.287047,0.559273,452.575,48,68,6.655515
ND4,0.217928,0.487998,1477.110,83,459,3.218105


In [24]:
ax=sns.regplot(data=metrics, x='mut_count', y='r2', scatter_kws={'s':10}, color='blue')
for gene, row in metrics.iterrows():
    plt.text(row['mut_count'], row['r2'], gene)

plt.ylabel('$R^2$', fontsize=12)
plt.xlabel('Количество мутаций', fontsize=12)
plt.grid(True)
ax.set_axisbelow(True)

from scipy.stats import pearsonr, spearmanr, shapiro

print(shapiro(metrics['r2']))
print(shapiro(metrics['mut_count']))
print(spearmanr(metrics['mut_count'], metrics['r2']))

ShapiroResult(statistic=np.float64(0.9711608594207852), pvalue=np.float64(0.9077340143649768))
ShapiroResult(statistic=np.float64(0.9254662115440425), pvalue=np.float64(0.29696152738273895))
SignificanceResult(statistic=np.float64(0.4340659340659341), pvalue=np.float64(0.1383423778882126))


In [25]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax=axes[0]
sns.regplot(data=metrics, x='mut_count', y='r2', 
               scatter_kws={'s':10}, color='blue', ax=ax)
for gene, row in metrics.iterrows():
    ax.text(row['mut_count'], row['r2'], gene)

ax.set_ylabel('$R^2$', fontsize=12)
ax.set_xlabel('Количество аминокислотных замен', fontsize=12)
ax.grid(True)
ax.set_axisbelow(True)

print('r2 vs mut_count:')
print(spearmanr(metrics['mut_count'], metrics['r2']))


ax=axes[1]
sns.regplot(data=metrics, x='mut_type_count', y='r2', 
            scatter_kws={'s':10}, color='blue', ax=ax)
for gene, row in metrics.iterrows():
    ax.text(row['mut_type_count'], row['r2'], gene)

ax.set_ylabel('$R^2$', fontsize=12)
ax.set_xlabel('Количество типов аминокислотных замен', fontsize=12)
ax.grid(True)
ax.set_axisbelow(True)
print('\nr2 vs mut_type_count:')
print(spearmanr(metrics['mut_type_count'], metrics['r2']))
plt.savefig('human_germline_r2_vs_mut_counts.pdf', bbox_inches='tight')

r2 vs mut_count:
SignificanceResult(statistic=np.float64(0.4340659340659341), pvalue=np.float64(0.1383423778882126))

r2 vs mut_type_count:
SignificanceResult(statistic=np.float64(0.506888975745196), pvalue=np.float64(0.0770812698847044))


In [26]:
ax=sns.regplot(data=metrics, x='mut_count_rel', y='r2', scatter_kws={'s':10}, color='blue')
for gene, row in metrics.iterrows():
    plt.text(row['mut_count_rel'], row['r2'], gene)

plt.ylabel('$R^2$', fontsize=12)
plt.xlabel('Относительное количество мутаций', fontsize=12)
plt.grid(True)
ax.set_axisbelow(True)

## Derive equilibium for each gene

In [27]:
import matplotlib

my_palette = sns.diverging_palette(12, 255, s=100, sep=50, as_cmap=True)

matplotlib.colormaps.register(my_palette, name="RedBlue3")

In [28]:
# Create a palette with 12 shades of gray and 1 red
colors = ['#808080'] * 12 + ['#FF0000']
custom_palette = LinearSegmentedColormap.from_list("GrayRed", colors, N=13)

# Register the palette
matplotlib.colormaps.register(name="GrayRed", cmap=custom_palette)

In [29]:
# Drawn after per-gene equilibrium frequencies are computed.


## compare obs and eq aa freqs

In [30]:
def plot_obs_vs_eq_aa_freqs(
        freqs_aa, title='', save_path=None, show=True
    ):
    plt.figure(figsize=(14, 4))
    ax = sns.barplot(freqs_aa, x='aa', y='eq_freq', color='lightgray', 
                    edgecolor=".55", label='Equilibrium', alpha=0.5)
    sns.barplot(data=freqs_aa, x='aa', y='obs_freq', alpha=0.5, 
                ax=ax, label='Observed', color='lightgreen',
                edgecolor=".55",)
    ax.set_title(title)
    ax.set_ylabel('Frequency', fontsize=14)
    ax.set_xlabel('Amino acid', fontsize=14)
    plt.legend()
    if save_path:
        plt.savefig(save_path)
    if show:
        plt.show()
    return ax

In [31]:
total_freqs = []
for gene in spectra_human_megatree.index.unique():
    _, cur_eq_aa_freqs = get_equilibrium_freqs(
        spectra_human_megatree.loc[gene], 'MutSpec', gc=2)
    cur_eq_aa_freqs.columns = ['aa', 'eq_freq']
    cur_obs_aa_freqs = aa_freqs[gene]
    cur_eq_aa_freqs['obs_freq'] = cur_eq_aa_freqs['aa'].map(
        {amino_acid_codes[x]:v for x, v in cur_obs_aa_freqs.items()}
    ).fillna(0)
    cur_eq_aa_freqs['obs_freq'] /= cur_eq_aa_freqs['obs_freq'].sum()
    cur_eq_aa_freqs['gene'] = gene
    # plot_obs_vs_eq_aa_freqs(cur_eq_aa_freqs.sort_values('eq_freq', ascending=False), title=gene)
    total_freqs.append(cur_eq_aa_freqs)
total_freqs = pd.concat(total_freqs)

lg_df = total_freqs.assign(flow=lambda d: d["eq_freq"] - d["obs_freq"])
plt.figure(figsize=(12, 5))
sns.barplot(
    lg_df, x='aa', y='flow', hue='gene',
    palette='GrayRed', edgecolor=".65",
    order=lg_df.query('gene == "CYTB"').sort_values('flow', ascending=False).aa,
)
plt.legend(
    loc='upper right', bbox_to_anchor=(1.12, 1.015), title='Ген мтДНК', fontsize=10)
plt.ylabel('Прирост частоты', fontsize=14)
plt.xlabel('')
plt.xticks(fontsize=12)
plt.savefig('./aa_flow_by_gene_megatree.pdf', bbox_inches='tight')
plt.show()


/tmp/ipykernel_85953/3469904144.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [32]:
total_freqs_eq = total_freqs.pivot(index='gene', columns='aa', values='eq_freq')
total_freqs_obs = total_freqs.pivot(index='gene', columns='aa', values='obs_freq')

In [33]:
total_freqs

,aa,eq_freq,obs_freq,gene
0,Thr,0.121260,0.115044,ATP6
1,Lys,0.097280,0.026549,ATP6
2,Pro,0.086547,0.061947,ATP6
3,Ser,0.078754,0.057522,ATP6
4,Asn,0.070867,0.048673,ATP6
...,...,...,...,...
15,Ile,0.017009,0.068966,ND6
16,Tyr,0.017009,0.063218,ND6
17,Phe,0.010795,0.045977,ND6
18,His,0.005993,0.000000,ND6


In [34]:
total_freqs_2genes = total_freqs.query('gene in ["ND6", "CYTB"]').set_index(['gene', 'aa'])
cytb_nd6_diff = total_freqs_2genes.loc['CYTB'] - total_freqs_2genes.loc['ND6']

sns.regplot(cytb_nd6_diff, y='eq_freq', x='obs_freq', scatter_kws={'s':10}, color='blue')
for aa, row in cytb_nd6_diff.iterrows():
    plt.text(row['obs_freq'], row['eq_freq'], aa)
plt.xlabel('Отличия наблюдаемых частот [CYTB - ND6]', fontsize=12)
plt.ylabel('Отличия равновесных частот [CYTB - ND6]', fontsize=12)
plt.grid(True)
plt.show()

print(pearsonr(
    cytb_nd6_diff['obs_freq'], 
    cytb_nd6_diff['eq_freq']
))

sns.barplot(cytb_nd6_diff.reset_index().sort_values('eq_freq'), x='aa', y='eq_freq', color='lightgray',)

PearsonRResult(statistic=np.float64(0.7573168301697818), pvalue=np.float64(0.00011041487809744476))


/tmp/ipykernel_85953/2227784.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Axes: xlabel='Отличия наблюдаемых частот [CYTB - ND6]', ylabel='Отличия равновесных частот [CYTB - ND6]'>

In [35]:
_ = total_freqs_2genes.reset_index().melt(id_vars=['gene', 'aa'], var_name='subset', value_name='freq')
_['lbl'] = _['gene'] + ' (' + _['subset'].replace({'eq_freq': 'eq', 'obs_freq': 'ref'}) + ')'

plt.figure(figsize=(12, 5))
ax = sns.barplot(
    _, x='aa', y='freq', hue='lbl', 
    palette=["#E68484", '#FF0000', "#88CE67", "#0F8E15"], 
    edgecolor=".01",
    hue_order=[
        'CYTB (ref)', 'CYTB (eq)', 
        'ND6 (ref)', 'ND6 (eq)',
    ],
    order=list(amino_acid_codes.values())[:-1],
)
plt.legend(
    loc='upper left',  title='', fontsize=13) # bbox_to_anchor=(1.12, 1.015),
plt.ylabel('Frequency', fontsize=14)
plt.xlabel('')
plt.xticks(fontsize=14)
plt.grid(axis='y')
ax.set_axisbelow(True)
plt.show()

/tmp/ipykernel_85953/2918837563.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [36]:
from sklearn.decomposition import PCA

# Perform PCA on total_freqs_obs
pca = PCA(n_components=2)
transformed_obs = pca.fit_transform(total_freqs_obs)

# Transform total_freqs_eq using the fitted PCA
transformed_eq = pca.transform(total_freqs_eq)

# Create a DataFrame for the transformed data
transformed_obs_df = pd.DataFrame(
    transformed_obs, columns=['PC1', 'PC2'], index=total_freqs_obs.index
)
transformed_eq_df = pd.DataFrame(
    transformed_eq, columns=['PC1', 'PC2'], index=total_freqs_eq.index
)

In [37]:
# Get explained variance ratio
explained_variance = pca.explained_variance_ratio_ * 100

# Create scatterplot with labels
plt.figure(figsize=(8, 5))
plt.scatter(transformed_obs_df['PC1'], transformed_obs_df['PC2'], 
    s=10, alpha=0.7, label='Observed', color='darkred')
plt.scatter(transformed_eq_df['PC1'], transformed_eq_df['PC2'], 
    s=10, alpha=0.7, label='Equilibrium', color='blue')

# Add labels for each point
for gene, (x, y) in transformed_obs_df.iterrows():
    plt.text(x, y, gene, fontsize=12, color='darkred')
for gene, (x, y) in transformed_eq_df.iterrows():
    plt.text(x, y, gene, fontsize=12, color='blue')

# Add axis labels with explained variance
plt.title('PCA of Amino Acid Frequencies: human mtDNA megatree')
plt.xlabel(f'PC1 ({explained_variance[0]:.2f}%)')
plt.ylabel(f'PC2 ({explained_variance[1]:.2f}%)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.savefig('./pca_aa_freqs_megatree.pdf', bbox_inches='tight')
plt.show()

/tmp/ipykernel_85953/2181340997.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [38]:
# PCA biplot: sample scores + amino acid loadings
loadings = pd.DataFrame(
    pca.components_.T,
    index=total_freqs_obs.columns,
    columns=['PC1', 'PC2']
 )

# Use symmetric limits so arrows and points are easier to compare
x_lim = max(
    abs(transformed_obs_df['PC1']).max(),
    abs(transformed_eq_df['PC1']).max()
 ) * 1.2
y_lim = max(
    abs(transformed_obs_df['PC2']).max(),
    abs(transformed_eq_df['PC2']).max()
 ) * 1.2


fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(
    transformed_obs_df['PC1'], transformed_obs_df['PC2'],
    s=25, alpha=0.75, label='Observed', color='darkred'
 )
ax.scatter(
    transformed_eq_df['PC1'], transformed_eq_df['PC2'],
    s=25, alpha=0.75, label='Equilibrium', color='blue'
 )

# Label sample points (genes)
for gene, row in transformed_obs_df.iterrows():
    ax.text(row['PC1'], row['PC2'], gene, fontsize=12, color='darkred', alpha=0.9)
for gene, row in transformed_eq_df.iterrows():
    ax.text(row['PC1'], row['PC2'], gene, fontsize=12, color='blue', alpha=0.9)

# Draw loading vectors for amino acids
arrow_scale = min(x_lim, y_lim) * 0.9
for aa, row in loadings.iterrows():
    x_end = row['PC1'] * arrow_scale
    y_end = row['PC2'] * arrow_scale
    ax.arrow(
        0, 0, x_end, y_end,
        color='gray', alpha=0.65, width=0.0,
        head_width=0.03 * arrow_scale, head_length=0.05 * arrow_scale,
        length_includes_head=True
    )
    ax.text(x_end * 1.05, y_end * 1.05, aa, color='black', fontsize=12, style='italic')

ax.axhline(0, color='black', lw=0.8, alpha=0.5)
ax.axvline(0, color='black', lw=0.8, alpha=0.5)
ax.set_xlim(-0.13, 0.22)
ax.set_ylim(-y_lim, 0.14)
ax.set_xlabel(f"PC1 ({explained_variance[0]:.2f}%)", fontsize=13)
ax.set_ylabel(f"PC2 ({explained_variance[1]:.2f}%)", fontsize=13)
ax.legend(loc='upper right', fontsize=12)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig('./pca_aa_freqs_megatree_biplot.pdf', bbox_inches='tight')
plt.show()

/tmp/ipykernel_85953/982356190.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [39]:
sns.lmplot(total_freqs, x='eq_freq', y='obs_freq', col='gene', col_wrap=5)

In [40]:
from scipy.spatial.distance import euclidean, cosine, cityblock

In [41]:
distances = []
for gene,cur_df  in total_freqs.groupby('gene'):
    x,y = cur_df.eq_freq.values, cur_df.obs_freq.values
    md = cityblock(x, y)
    ed = euclidean(x, y)
    cd = cosine(x, y)
    distances.append([gene, md, ed, cd])
distances = pd.DataFrame(distances, columns=['gene', 'manhattan', 'euclidean', 'cosine'])
distances = distances.set_index('gene')

In [42]:
distances.manhattan.sort_values(ascending=False).plot.barh(
    color=['gray']*11+['darkred']+['gray'], figsize=(3, 5))
# plt.xlabel('Manhattan distance to equilibtrium')
plt.xlabel('Расстояние Манхэттена\nдо равновесия')
plt.ylabel('Ген')
plt.xlim(0.55, 0.91)
plt.show()

/tmp/ipykernel_85953/1901383565.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
